# Error Analysis — IMRaD Introduction Classifier

This notebook performs qualitative error analysis for the four fine-tuned BERT classifiers described in the paper. For each model it:

1. Reproduces the **exact same test split** used in training (deterministic `random_state=42`).
2. Runs inference via the published Hugging Face SavedModels.
3. Identifies the **top confused class pairs** (by error count).
4. Prints **3 representative misclassified sentences** per confused pair for qualitative inspection.

Models analysed:
- **Model 1**: Overall move classifier (Move 0 / Move 1 / Move 2)
- **Model 2**: Move 0 sub-move specialist (2 classes)
- **Model 3**: Move 1 sub-move specialist (4 classes)
- **Model 4**: Move 2 sub-move specialist (5 classes)

> **Reproducibility note:** `df.sample(frac=0.8, random_state=42)` is deterministic given the same row order. The Hugging Face dataset preserves the original row order, so the test split here is identical to the one used during training.

## 1. Install dependencies

In [1]:
!pip install -q tensorflow-text==2.15.0
!pip install -q tf-models-official==2.15.0
!pip install -q datasets huggingface_hub

ERROR: Could not find a version that satisfies the requirement tensorflow-text==2.15.0 (from versions: 2.18.1, 2.19.0rc0, 2.19.0, 2.20.0, 2.20.1)
ERROR: No matching distribution found for tensorflow-text==2.15.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
ERROR: Could not find a version that satisfies the requirement tensorflow-text~=2.15.0 (from tf-models-official) (from versions: 2.18.1, 2.19.0rc0, 2.19.0, 2.20.0, 2.20.1)
ERROR: No matching distribution found for tensorflow-text~=2.15.0


## 2. Imports

In [2]:
import numpy as np
import pandas as pd

import tensorflow as tf
import tensorflow_text  # required to deserialise TF Hub BERT layers
from sklearn import preprocessing
from sklearn.metrics import classification_report
from datasets import load_dataset, Features, Value
from huggingface_hub import snapshot_download

print('TF:', tf.__version__)
tf.get_logger().setLevel('ERROR')

TF: 2.20.0


## 3. Constants

In [3]:
SEED       = 42
BATCH      = 32
DATASET_ID = 'stormsidali2001/IMRAD-introduction-sentences-moves-sub-moves-dataset'

MODEL_REPOS = {
    1: 'stormsidali2001/IMRAD_introduction_moves_classifier',
    2: 'stormsidali2001/IMRAD-introduction-move-zero-sub-moves-classifier',
    3: 'stormsidali2001/IMRAD-introduction-move-one-sub-moves-classifier',
    4: 'stormsidali2001/IMRAD-introduction-move-two-sub-moves-classifier',
}

## 4. Load and preprocess dataset

The `move_sub_move_gemini` column is forced to string type to avoid Arrow parse errors on dirty rows (LaTeX tokens not cleaned during dataset construction). Rows that cannot be parsed to a valid integer label are dropped.

In [4]:
features = Features({
    'sentence':            Value('string'),
    'move_sub_move_gemini': Value('string'),
})
hf_ds  = load_dataset(DATASET_ID, split='train', features=features)
raw_df = hf_ds.to_pandas()
print('Loaded:', raw_df.shape)

def safe_int(x):
    try:
        return int(float(str(x).strip()))
    except (ValueError, TypeError):
        return -1

raw_df['gemini_move'] = raw_df['move_sub_move_gemini'].apply(safe_int)
raw_df['sentence']    = raw_df['sentence'].apply(lambda x: str(x).lower())

full_df = raw_df[raw_df['gemini_move'] != -1].copy()
print('After filtering:', full_df.shape)
print(full_df['gemini_move'].value_counts().sort_index())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

dataset.csv:   0%|          | 0.00/75.1M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Loaded: (200328, 2)
After filtering: (169729, 3)
gemini_move
0    56468
1    55604
2    57657
Name: count, dtype: int64


## 5. Helpers

`make_split` replicates the exact train/val/test split from the training notebooks.  
`load_saved_model` loads a TF SavedModel from Hugging Face via `serving_default`.  
`batch_predict` runs batched CPU/GPU inference.  
`show_errors` extracts the top confused pairs and prints representative misclassified sentences.

In [5]:
def make_split(df):
    train     = df.sample(frac=0.8, random_state=SEED)
    remainder = df.drop(train.index)
    val       = remainder.sample(frac=0.5, random_state=SEED)
    test      = remainder.drop(val.index)
    return train, val, test


def load_saved_model(repo_id):
    path   = snapshot_download(repo_id)
    loaded = tf.saved_model.load(path)
    infer  = loaded.signatures['serving_default']
    input_key  = list(infer.structured_input_signature[1].keys())[0]
    output_key = list(infer.structured_outputs.keys())[0]
    print(f'  Loaded: input={input_key!r}  output={output_key!r}')
    return infer, input_key, output_key


def batch_predict(infer, input_key, output_key, texts):
    all_out = []
    for i in range(0, len(texts), BATCH):
        batch = tf.constant(texts[i:i+BATCH])
        out   = infer(**{input_key: batch})[output_key].numpy()
        all_out.append(out)
    return np.concatenate(all_out, axis=0)


def show_errors(test_df, y_true, y_pred, label_names,
                top_n_pairs=3, examples_per_pair=3, max_chars=300):
    """Print top confused pairs and representative misclassified sentences."""
    df = test_df.copy().reset_index(drop=True)
    df['true'] = y_true
    df['pred'] = y_pred
    errors = df[df['true'] != df['pred']]

    total   = len(df)
    n_err   = len(errors)
    print(f'Total errors: {n_err} / {total}  ({100*n_err/total:.1f}%)')
    print(f'Overall accuracy: {100*(total-n_err)/total:.2f}%\n')

    pair_counts = (
        errors.groupby(['true', 'pred'])
        .size()
        .sort_values(ascending=False)
    )

    print('Top confused pairs:')
    for (t, p), cnt in pair_counts.items():
        pct = 100 * cnt / total
        print(f'  True={label_names[t]:40s}  Pred={label_names[p]:40s}  n={cnt:4d} ({pct:.1f}%)')
    print()

    for (t, p), cnt in pair_counts.head(top_n_pairs).items():
        tname = label_names[t]
        pname = label_names[p]
        print('=' * 80)
        print(f'True: {tname}')
        print(f'Pred: {pname}  ({cnt} cases)')
        print('=' * 80)
        sample = (
            errors[(errors['true'] == t) & (errors['pred'] == p)]
            ['sentence']
            .sample(min(examples_per_pair, cnt), random_state=SEED)
        )
        for i, s in enumerate(sample, 1):
            print(f'  [{i}] {s[:max_chars]}')
        print()

## 6. Model 1 — Overall Move Classifier (3 classes)

In [6]:
_, _, test1 = make_split(full_df)
print(f'Test set: {len(test1)} sentences')

infer1, ik1, ok1 = load_saved_model(MODEL_REPOS[1])
raw1    = batch_predict(infer1, ik1, ok1, test1['sentence'].to_numpy())
y_pred1 = np.argmax(raw1, axis=1)
y_true1 = test1['gemini_move'].values

print(classification_report(y_true1, y_pred1,
      target_names=['Move 0', 'Move 1', 'Move 2']))

label_names1 = {
    0: 'Move 0 (Establish Research Territory)',
    1: 'Move 1 (Establish a Niche)',
    2: 'Move 2 (Occupy the Niche)',
}
show_errors(test1, y_true1, y_pred1, label_names1)

Test set: 16973 sentences


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

  Loaded: input='text'  output='classifier'
              precision    recall  f1-score   support

      Move 0       0.98      0.98      0.98      5669
      Move 1       0.99      0.98      0.98      5431
      Move 2       0.98      0.99      0.99      5873

    accuracy                           0.98     16973
   macro avg       0.98      0.98      0.98     16973
weighted avg       0.98      0.98      0.98     16973

Total errors: 303 / 16973  (1.8%)
Overall accuracy: 98.21%

Top confused pairs:
  True=Move 1 (Establish a Niche)                Pred=Move 0 (Establish Research Territory)     n=  76 (0.4%)
  True=Move 0 (Establish Research Territory)     Pred=Move 1 (Establish a Niche)                n=  65 (0.4%)
  True=Move 0 (Establish Research Territory)     Pred=Move 2 (Occupy the Niche)                 n=  61 (0.4%)
  True=Move 1 (Establish a Niche)                Pred=Move 2 (Occupy the Niche)                 n=  48 (0.3%)
  True=Move 2 (Occupy the Niche)                 Pred=M

## 7. Model 2 — Move 0 Sub-move Specialist (2 classes)

In [7]:
df2 = full_df[full_df['gemini_move'] == 0].copy()
df2 = df2.sort_values(by='move_sub_move_gemini')
le2 = preprocessing.LabelEncoder()
df2['encoded'] = le2.fit_transform(df2['move_sub_move_gemini'])

_, _, test2 = make_split(df2)
print(f'Test set: {len(test2)} sentences  |  Classes: {list(le2.classes_)}')

infer2, ik2, ok2 = load_saved_model(MODEL_REPOS[2])
raw2    = batch_predict(infer2, ik2, ok2, test2['sentence'].to_numpy())
y_pred2 = np.argmax(raw2, axis=1)
y_true2 = test2['encoded'].values

print(classification_report(y_true2, y_pred2,
      target_names=[str(c) for c in le2.classes_]))

label_names2 = {
    0: '0.0: Establish importance/relevance',
    1: '0.1: Review prior research',
}
show_errors(test2, y_true2, y_pred2, label_names2)

Test set: 5647 sentences  |  Classes: ['0.0', '0.1']


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

  Loaded: input='text'  output='classifier'
              precision    recall  f1-score   support

         0.0       0.91      0.87      0.89      2765
         0.1       0.88      0.92      0.90      2882

    accuracy                           0.90      5647
   macro avg       0.90      0.90      0.90      5647
weighted avg       0.90      0.90      0.90      5647

Total errors: 587 / 5647  (10.4%)
Overall accuracy: 89.61%

Top confused pairs:
  True=0.0: Establish importance/relevance       Pred=0.1: Review prior research                n= 346 (6.1%)
  True=0.1: Review prior research                Pred=0.0: Establish importance/relevance       n= 241 (4.3%)

True: 0.0: Establish importance/relevance
Pred: 0.1: Review prior research  (346 cases)
  [1] a substantial body of research has demonstrated the critical impact of x on various outcomes.
  [2] extensive research has demonstrated the crucial role played by x in the field, highlighting its significance and relevance.
  [3] exte

## 8. Model 3 — Move 1 Sub-move Specialist (4 classes)

In [8]:
df3 = full_df[full_df['gemini_move'] == 1].copy()
df3 = df3.sort_values(by='move_sub_move_gemini')
le3 = preprocessing.LabelEncoder()
df3['encoded'] = le3.fit_transform(df3['move_sub_move_gemini'])

_, _, test3 = make_split(df3)
print(f'Test set: {len(test3)} sentences  |  Classes: {list(le3.classes_)}')

infer3, ik3, ok3 = load_saved_model(MODEL_REPOS[3])
raw3    = batch_predict(infer3, ik3, ok3, test3['sentence'].to_numpy())
y_pred3 = np.argmax(raw3, axis=1)
y_true3 = test3['encoded'].values

print(classification_report(y_true3, y_pred3,
      target_names=[str(c) for c in le3.classes_]))

label_names3 = {
    0: '1.0: Claim flaw in prior work',
    1: '1.1: Highlight a gap',
    2: '1.2: Raise an unclear question',
    3: '1.3: Extend prior research',
}
show_errors(test3, y_true3, y_pred3, label_names3)

Test set: 5561 sentences  |  Classes: ['1.0', '1.1', '1.2', '1.3']


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

  Loaded: input='text'  output='classifier'
              precision    recall  f1-score   support

         1.0       0.97      0.97      0.97      1454
         1.1       0.92      0.92      0.92      1422
         1.2       0.94      0.93      0.94      1317
         1.3       0.95      0.96      0.96      1368

    accuracy                           0.95      5561
   macro avg       0.95      0.95      0.95      5561
weighted avg       0.95      0.95      0.95      5561

Total errors: 302 / 5561  (5.4%)
Overall accuracy: 94.57%

Top confused pairs:
  True=1.2: Raise an unclear question            Pred=1.1: Highlight a gap                      n=  55 (1.0%)
  True=1.1: Highlight a gap                      Pred=1.2: Raise an unclear question            n=  52 (0.9%)
  True=1.1: Highlight a gap                      Pred=1.0: Claim flaw in prior work             n=  38 (0.7%)
  True=1.3: Extend prior research                Pred=1.1: Highlight a gap                      n=  35 (0.6%)
  

## 9. Model 4 — Move 2 Sub-move Specialist (5 classes)

In [9]:
df4 = full_df[full_df['gemini_move'] == 2].copy()
df4 = df4.sort_values(by='move_sub_move_gemini')
le4 = preprocessing.LabelEncoder()
df4['encoded'] = le4.fit_transform(df4['move_sub_move_gemini'])

_, _, test4 = make_split(df4)
print(f'Test set: {len(test4)} sentences  |  Classes: {list(le4.classes_)}')

infer4, ik4, ok4 = load_saved_model(MODEL_REPOS[4])
raw4    = batch_predict(infer4, ik4, ok4, test4['sentence'].to_numpy())
y_pred4 = np.argmax(raw4, axis=1)
y_true4 = test4['encoded'].values

print(classification_report(y_true4, y_pred4,
      target_names=[str(c) for c in le4.classes_]))

label_names4 = {
    0: '2.0: State purpose/aim',
    1: '2.1: State hypothesis',
    2: '2.2: Share findings',
    3: '2.3: Elaborate value',
    4: '2.4: Outline structure',
}
show_errors(test4, y_true4, y_pred4, label_names4)

Test set: 5765 sentences  |  Classes: ['2.0', '2.1', '2.2', '2.3', '2.4']


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

  Loaded: input='text'  output='classifier'
              precision    recall  f1-score   support

         2.0       0.97      0.99      0.98      1398
         2.1       0.99      0.97      0.98      1133
         2.2       0.95      0.96      0.96      1164
         2.3       0.96      0.95      0.95      1172
         2.4       1.00      1.00      1.00       898

    accuracy                           0.97      5765
   macro avg       0.97      0.97      0.97      5765
weighted avg       0.97      0.97      0.97      5765

Total errors: 158 / 5765  (2.7%)
Overall accuracy: 97.26%

Top confused pairs:
  True=2.3: Elaborate value                      Pred=2.2: Share findings                       n=  54 (0.9%)
  True=2.2: Share findings                       Pred=2.3: Elaborate value                      n=  36 (0.6%)
  True=2.1: State hypothesis                     Pred=2.0: State purpose/aim                    n=  29 (0.5%)
  True=2.3: Elaborate value                      Pred=2.0:

## 10. Save output to file

Run this cell to capture all printed output to a text file, then download it.

In [10]:
import io, contextlib

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    print('=== MODEL 1 ===')
    show_errors(test1, y_true1, y_pred1, label_names1)
    print('\n=== MODEL 2 ===')
    show_errors(test2, y_true2, y_pred2, label_names2)
    print('\n=== MODEL 3 ===')
    show_errors(test3, y_true3, y_pred3, label_names3)
    print('\n=== MODEL 4 ===')
    show_errors(test4, y_true4, y_pred4, label_names4)

output_text = buf.getvalue()
with open('error_analysis_output.txt', 'w') as f:
    f.write(output_text)
print('Saved to error_analysis_output.txt')
print(output_text[:500], '...')

Saved to error_analysis_output.txt
=== MODEL 1 ===
Total errors: 303 / 16973  (1.8%)
Overall accuracy: 98.21%

Top confused pairs:
  True=Move 1 (Establish a Niche)                Pred=Move 0 (Establish Research Territory)     n=  76 (0.4%)
  True=Move 0 (Establish Research Territory)     Pred=Move 1 (Establish a Niche)                n=  65 (0.4%)
  True=Move 0 (Establish Research Territory)     Pred=Move 2 (Occupy the Niche)                 n=  61 (0.4%)
  True=Move 1 (Establish a Niche)                Pred=Move 2 (Occupy the N ...


In [11]:
from google.colab import files
files.download('error_analysis_output.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>